In [3]:
# IrisPathQ Route Optimizer v0.2

#Classical preprocessing for quantum route optimization

#- Fuel and Distance calculations
#- Cost matrix construction

import subprocess
import os

print("IrisPathQ Route Optimizer v0.2")
print("Setting up C compilation environment...\n")

# gcc verification
result = subprocess.run(["gcc", "--version"], capture_output=True, text=True)
print(result.stdout.split('\n')[0])
print("\nReady for next step")

IrisPathQ Route Optimizer v0.2
Setting up C compilation environment...

gcc (GCC) 15.1.0

Ready for next step


In [5]:
%%writefile data_structures.h
/**
 * IrisPathQ Route Optimizer
 * Data structures for flights, routes, and optimization
*/

#ifndef DATA_STRUCTURES_H
#define DATA_STRUCTURES_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

#define MAX_FLIGHTS 500
#define MAX_WAYPOINTS 1000
#define MAX_ROUTES_PER_FLIGHT 10
#define MAX_WAYPOINTS_PER_ROUTE 50
#define EARTH_RADIUS_KM 6371.0
#define M_PI 3.14159265358979323846

typedef struct {
    char id[10];
    char name[50];
    double latitude;
    double longitude;
} Waypoint;

typedef struct {
    double latitude;
    double longitude;
    double wind_speed;
    double wind_direction;
    char weather_type[20];
} Weather;

typedef struct {
    char type[10];
    double cruise_speed;
    double fuel_burn_rate;
    double max_range;
} AircraftType;

typedef struct {
    char flight_id[10];
    char origin[10];
    char destination[10];
    char departure_time[10];
    AircraftType aircraft;
} Flight;

typedef struct {
    int waypoint_indices[MAX_WAYPOINTS_PER_ROUTE];
    int num_waypoints;
    double total_distance;
    double fuel_cost;
    double time_cost;
    int conflict_count;
} Route;

typedef struct {
    Flight flights[MAX_FLIGHTS];
    int num_flights;
    
    Waypoint waypoints[MAX_WAYPOINTS];
    int num_waypoints;
    
    Weather weather_cells[100];
    int num_weather_cells;
    
    Route routes[MAX_FLIGHTS][MAX_ROUTES_PER_FLIGHT];
    int num_routes_per_flight[MAX_FLIGHTS];
} ProblemInstance;

double calculate_distance(double lat1, double lon1, double lat2, double lon2);
double calculate_fuel(Route *route, AircraftType *aircraft);
void print_route(Route *route, Waypoint *waypoints);
int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells);

int load_flights(const char *filename, ProblemInstance *problem);
int load_waypoints(const char *filename, ProblemInstance *problem);
int load_weather(const char *filename, ProblemInstance *problem);
int generate_alternative_routes(ProblemInstance *problem, int flight_idx, int num_routes);
void inject_conflicts(ProblemInstance *problem);
void build_cost_matrix(ProblemInstance *problem, double **cost_matrix, int *matrix_size);
void export_cost_matrix(double *cost_matrix, int matrix_size, const char *filename);
void export_full_matrix(double *cost_matrix, int matrix_size, const char *filename);

#endif

Overwriting data_structures.h


In [6]:
    %%writefile utils.c
    /**
     * IrisPathQ Route Optimizer
     * Distance and fuel calculations
    */
    
    #include "data_structures.h"
    
    double calculate_distance(double lat1, double lon1, double lat2, double lon2) {
        //degree to radian conversion
        double lat1_rad = lat1 * M_PI / 180.0;
        double lat2_rad = lat2 * M_PI / 180.0;
        double delta_lat = (lat2 - lat1) * M_PI / 180.0;
        double delta_lon = (lon2 - lon1) * M_PI / 180.0;
    
        //haversineformula 
        double a = sin(delta_lat / 2.0) * sin(delta_lat / 2.0) +
                   cos(lat1_rad) * cos(lat2_rad) *
                   sin(delta_lon / 2.0) * sin(delta_lon / 2.0);
        
        double c = 2.0 * atan2(sqrt(a), sqrt(1.0 - a));
        double distance_km = EARTH_RADIUS_KM * c;
        
        return distance_km / 1.852;
    }
    
    double calculate_fuel(Route *route, AircraftType *aircraft) {
        double time_hours = route->total_distance / aircraft->cruise_speed;
        return time_hours * aircraft->fuel_burn_rate;
    }
    
    // For conflicts due to weather, but rigth now no conflicts-- weather.csv has no thunderstorm as input
    int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells) {
        for (int i = 0; i < num_cells; i++) {
            if (strcmp(weather_cells[i].weather_type, "THUNDERSTORM") == 0) {
                double dist = calculate_distance(lat, lon, 
                                                weather_cells[i].latitude, 
                                                weather_cells[i].longitude);
                if (dist < 50.0) {
                    return 1;
                }
            }
        }
        return 0;
    }
    
    void print_route(Route *route, Waypoint *waypoints) {
        printf("Route: ");
        for (int i = 0; i < route->num_waypoints; i++) {
            printf("%s ", waypoints[route->waypoint_indices[i]].id);
            if (i < route->num_waypoints - 1) printf("-> ");
        }
        printf("\n");
        printf("Distance: %.2f nm, Fuel: %.2f kg, Time: %.2f hrs\n",
               route->total_distance, route->fuel_cost, route->time_cost);
    }

Overwriting utils.c


In [7]:
%%writefile data_loader.c
/**
 * IrisPathQ Route Optimizer
 * Load flights, waypoints, and weather data from CSV files
*/

#include "data_structures.h"

int load_flights(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_flights = 0;
    while (fgets(line, sizeof(line), file) && problem->num_flights < MAX_FLIGHTS) {
        Flight *f = &problem->flights[problem->num_flights];
        
        sscanf(line, "%[^,],%[^,],%[^,],%[^,],%s",
               f->flight_id, f->origin, f->destination, 
               f->departure_time, f->aircraft.type);
        
        if (strcmp(f->aircraft.type, "B737") == 0) {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2500.0;
            f->aircraft.max_range = 3000.0;
        } else if (strcmp(f->aircraft.type, "A320") == 0) {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2400.0;
            f->aircraft.max_range = 3200.0;
        } else {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2500.0;
            f->aircraft.max_range = 3000.0;
        }
        
        problem->num_flights++;
    }
    
    fclose(file);
    printf("Loaded %d flights\n", problem->num_flights);
    return problem->num_flights;
}

int load_waypoints(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_waypoints = 0;
    while (fgets(line, sizeof(line), file) && problem->num_waypoints < MAX_WAYPOINTS) {
        Waypoint *w = &problem->waypoints[problem->num_waypoints];
        
        sscanf(line, "%[^,],%lf,%lf,%s",
               w->id, &w->latitude, &w->longitude, w->name);
        
        problem->num_waypoints++;
    }
    
    fclose(file);
    printf("Loaded %d waypoints\n", problem->num_waypoints);
    return problem->num_waypoints;
}

int load_weather(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_weather_cells = 0;
    while (fgets(line, sizeof(line), file) && problem->num_weather_cells < 100) {
        Weather *w = &problem->weather_cells[problem->num_weather_cells];
        
        sscanf(line, "%lf,%lf,%lf,%lf,%s",
               &w->latitude, &w->longitude, &w->wind_speed, 
               &w->wind_direction, w->weather_type);
        
        problem->num_weather_cells++;
    }
    
    fclose(file);
    printf("Loaded %d weather cells\n", problem->num_weather_cells);
    return problem->num_weather_cells;
}

Overwriting data_loader.c


In [16]:
%%writefile astar.c
#include "data_structures.h"
#include <float.h>

// Priority queue node for A*
typedef struct PQNode {
    int waypoint_index;
    double g_cost;  // Actual cost from start
    double h_cost;  // Heuristic cost to goal
    double f_cost;  // g + h
    int parent_index;
    struct PQNode *next;
} PQNode;

// Priority queue (min-heap as linked list for simplicity)
typedef struct {
    PQNode *head;
    int size;
} PriorityQueue;

// Initialize priority queue
void pq_init(PriorityQueue *pq) {
    pq->head = NULL;
    pq->size = 0;
}

// Insert into priority queue (sorted by f_cost)
void pq_insert(PriorityQueue *pq, int waypoint_index, double g_cost, double h_cost, int parent_index) {
    PQNode *new_node = (PQNode*)malloc(sizeof(PQNode));
    new_node->waypoint_index = waypoint_index;
    new_node->g_cost = g_cost;
    new_node->h_cost = h_cost;
    new_node->f_cost = g_cost + h_cost;
    new_node->parent_index = parent_index;
    new_node->next = NULL;
    
    // Insert in sorted order
    if (pq->head == NULL || pq->head->f_cost > new_node->f_cost) {
        new_node->next = pq->head;
        pq->head = new_node;
    } else {
        PQNode *current = pq->head;
        while (current->next != NULL && current->next->f_cost <= new_node->f_cost) {
            current = current->next;
        }
        new_node->next = current->next;
        current->next = new_node;
    }
    pq->size++;
}

// Extract minimum from priority queue
PQNode pq_extract_min(PriorityQueue *pq) {
    if (pq->head == NULL) {
        PQNode empty = {-1, 0, 0, 0, -1, NULL};
        return empty;
    }
    
    PQNode *min_node = pq->head;
    PQNode result = *min_node;
    pq->head = min_node->next;
    free(min_node);
    pq->size--;
    
    return result;
}

// Check if priority queue is empty
int pq_is_empty(PriorityQueue *pq) {
    return pq->size == 0;
}

// Free priority queue
void pq_free(PriorityQueue *pq) {
    while (pq->head != NULL) {
        PQNode *temp = pq->head;
        pq->head = pq->head->next;
        free(temp);
    }
}

// Heuristic: Straight-line distance (great circle)
double heuristic(Waypoint *from, Waypoint *to) {
    return calculate_distance(from->latitude, from->longitude,to->latitude, to->longitude);
}

// Check if waypoint is in weather hazard (from utils.c)
extern int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells);

// A* pathfinding algorithm
int astar_find_route(ProblemInstance *problem, int origin_idx, int destination_idx, Route *route) {
    Waypoint *waypoints = problem->waypoints;
    int num_waypoints = problem->num_waypoints;
    Weather *weather = problem->weather_cells;
    int num_weather = problem->num_weather_cells;
    
    // Initialize data structures
    double g_cost[MAX_WAYPOINTS];
    int parent[MAX_WAYPOINTS];
    int visited[MAX_WAYPOINTS];
    
    for (int i = 0; i < num_waypoints; i++) {
        g_cost[i] = DBL_MAX;
        parent[i] = -1;
        visited[i] = 0;
    }
    
    g_cost[origin_idx] = 0.0;
    
    // Priority queue
    PriorityQueue pq;
    pq_init(&pq);
    
    double h = heuristic(&waypoints[origin_idx], &waypoints[destination_idx]);
    pq_insert(&pq, origin_idx, 0.0, h, -1);
    
    int found = 0;
    
    // A* main loop
    while (!pq_is_empty(&pq)) {
        PQNode current = pq_extract_min(&pq);
        int current_idx = current.waypoint_index;
        
        // Goal reached
        if (current_idx == destination_idx) {
            found = 1;
            break;
        }
        
        // Skip if already visited
        if (visited[current_idx]) {
            continue;
        }
        visited[current_idx] = 1;
        
        // Explore neighbors (for simplicity, consider all waypoints as potential neighbors)
        // Airways to be used here 
        for (int neighbor_idx = 0; neighbor_idx < num_waypoints; neighbor_idx++) {
            if (neighbor_idx == current_idx || visited[neighbor_idx]) {
                continue;
            }
            
            // Calculate distance to neighbor
            double edge_cost = calculate_distance(
                waypoints[current_idx].latitude, waypoints[current_idx].longitude,
                waypoints[neighbor_idx].latitude, waypoints[neighbor_idx].longitude
            );
            
            // Skip if too far 
            if (edge_cost > 3000.0) {  // Max 500nm between waypoints too less
                continue;
            }
            
            // Add penalty for weather hazards
            if (is_in_hazard(waypoints[neighbor_idx].latitude, 
                           waypoints[neighbor_idx].longitude,
                           weather, num_weather)) {
                edge_cost *= 2.0;  // Double the cost for hazardous areas
            }
            
            // Calculate tentative g_cost
            double tentative_g = g_cost[current_idx] + edge_cost;
            
            // Update if better path found
            if (tentative_g < g_cost[neighbor_idx]) {
                g_cost[neighbor_idx] = tentative_g;
                parent[neighbor_idx] = current_idx;
                
                double h = heuristic(&waypoints[neighbor_idx], &waypoints[destination_idx]);
                pq_insert(&pq, neighbor_idx, tentative_g, h, current_idx);
            }
        }
    }
    
    pq_free(&pq);
    
    if (!found) {
        printf("No route found from %s to %s\n", 
               waypoints[origin_idx].id, waypoints[destination_idx].id);
        return -1;
    }
    
    // Reconstruct path
    int path[MAX_WAYPOINTS_PER_ROUTE];
    int path_length = 0;
    int current = destination_idx;
    
    while (current != -1) {
        path[path_length++] = current;
        current = parent[current];
    }
    
    // Reverse path (it's built backwards)
    route->num_waypoints = path_length;
    for (int i = 0; i < path_length; i++) {
        route->waypoint_indices[i] = path[path_length - 1 - i];
    }
    
    // Calculate total distance and fuel
    route->total_distance = g_cost[destination_idx];
    route->conflict_count = 0;  // Will be calculated later  no conflicts yet 
    
    return 0;
}

// Generate multiple alternative routes by adding randomness
int generate_alternative_routes(ProblemInstance *problem, int flight_idx, int num_routes) {
    Flight *flight = &problem->flights[flight_idx];
    
    // Find origin and destination waypoint indices
    int origin_idx = -1, dest_idx = -1;
    for (int i = 0; i < problem->num_waypoints; i++) {
        if (strcmp(problem->waypoints[i].id, flight->origin) == 0) {
            origin_idx = i;
        }
        if (strcmp(problem->waypoints[i].id, flight->destination) == 0) {
            dest_idx = i;
        }
    }
    
    if (origin_idx == -1 || dest_idx == -1) {
        printf("Error: Origin or destination not found for flight %s\n", flight->flight_id);
        return -1;
    }
    
    printf("Generating %d routes for %s: %s -> %s\n", 
           num_routes, flight->flight_id, flight->origin, flight->destination);
    
    // Generate multiple routes
    for (int r = 0; r < num_routes && r < MAX_ROUTES_PER_FLIGHT; r++) {
        Route *route = &problem->routes[flight_idx][r];
        
        // First route: standard A*
        if (r == 0) {
            if (astar_find_route(problem, origin_idx, dest_idx, route) == 0) {
                // Calculate fuel and time
                route->fuel_cost = calculate_fuel(route, &flight->aircraft);
                route->time_cost = route->total_distance / flight->aircraft.cruise_speed;
                
                printf("  Route %d: ", r);
                print_route(route, problem->waypoints);
            }
        } else {
            // Alternative routes: modify cost function slightly
            // intermediate waypoints
            // For now, just create variations
            
            // Simple variation: copy route 0 and mark as alternative
            *route = problem->routes[flight_idx][0];
            
            // Adding small random cost variation (simulating different paths)
            route->fuel_cost *= (1.0 + (r * 0.02));  // 2% increase per alternative
            route->time_cost *= (1.0 + (r * 0.02));
            
            printf("  Route %d: (variant) ", r);
            print_route(route, problem->waypoints);
        }
        
        problem->num_routes_per_flight[flight_idx]++;
    }
    
    return problem->num_routes_per_flight[flight_idx];
}
    //Artificial Conflicts for now 
    //inserts that waypoint into the second position of Flight N+1(s) Route 0. e.g 
    //Flight 1 -> A->B->C->D
    //Flight2 -> F->G->H->I
    //fucntion will make flight 2 path t  go via B new route F2: F->A->GHI
    //for Route 0 of every plane   
    void inject_conflicts(ProblemInstance *problem) {
    printf("\nInjecting strategic conflicts for quantum demonstration...\n");
    
    int conflicts_added = 0;
    
    // Create chain of conflicts: Flight Ns Route 0 conflicts with Flight N+1s Route 0
    for (int f = 0; f < problem->num_flights - 1; f++) {
        Route *current = &problem->routes[f][0];
        Route *next = &problem->routes[f+1][0];
        
        if (current->num_waypoints > 1 && next->num_waypoints < MAX_WAYPOINTS_PER_ROUTE - 1) {
            // Share a waypoint
            int shared_wp = current->waypoint_indices[current->num_waypoints - 2];
            
            // Insert into next route
            for (int w = next->num_waypoints; w > 1; w--) {
                next->waypoint_indices[w] = next->waypoint_indices[w-1];
            }
            next->waypoint_indices[1] = shared_wp;
            next->num_waypoints++;
            
            conflicts_added++;
            printf("  Flight %d Route 0 <-> Flight %d Route 0\n", f, f+1);
        }
    }
    
    printf("Added %d strategic conflicts\n\n", conflicts_added);
}

// Build cost matrix for QUBO encoding
void build_cost_matrix(ProblemInstance *problem, double **cost_matrix, int *matrix_size) {
    // Calculate total number of variables
    int total_vars = 0;
    for (int i = 0; i < problem->num_flights; i++) {
        total_vars += problem->num_routes_per_flight[i];
    }
    
    *matrix_size = total_vars;
    
    // Allocate matrix
    *cost_matrix = (double*)calloc(total_vars * total_vars, sizeof(double));
    
    printf("\nBuilding cost matrix (%d x %d)...\n", total_vars, total_vars);
    
    int var_index = 0;
    
    // Fill diagonal with route costs
    for (int flight = 0; flight < problem->num_flights; flight++) {
        for (int route = 0; route < problem->num_routes_per_flight[flight]; route++) {
            double fuel_cost = problem->routes[flight][route].fuel_cost;
            (*cost_matrix)[var_index * total_vars + var_index] = fuel_cost;
            var_index++;
        }
    }
    
    // Fill off-diagonal with conflict penalties
    int var_i = 0;
    for (int flight_i = 0; flight_i < problem->num_flights; flight_i++) {
        for (int route_i = 0; route_i < problem->num_routes_per_flight[flight_i]; route_i++) {
            
            int var_j = 0;
            for (int flight_j = 0; flight_j < problem->num_flights; flight_j++) {
                if (flight_i == flight_j) {
                    var_j += problem->num_routes_per_flight[flight_j];
                    continue;  // Same flight, no conflict
                }
                
                for (int route_j = 0; route_j < problem->num_routes_per_flight[flight_j]; route_j++) {
                    // Check if routes conflict (simple check: any shared waypoints)
                    Route *r1 = &problem->routes[flight_i][route_i];
                    Route *r2 = &problem->routes[flight_j][route_j];
                    
                    int conflict = 0;
                    for (int w1 = 0; w1 < r1->num_waypoints; w1++) {
                        for (int w2 = 0; w2 < r2->num_waypoints; w2++) {
                            if (r1->waypoint_indices[w1] == r2->waypoint_indices[w2]) {
                                conflict = 1;
                                break;
                            }
                        }
                        if (conflict) break;
                    }
                    
                    if (conflict) {
                        // Add penalty for conflict
                        (*cost_matrix)[var_i * total_vars + var_j] = 10000.0;  // Large penalty
                        (*cost_matrix)[var_j * total_vars + var_i] = 10000.0;  // Symmetric
                    }
                    
                    var_j++;
                }
            }
            var_i++;
        }
    }
    
    printf("Cost matrix built successfully.\n");
}

// Export cost matrix to file (for quantum processing)
void export_cost_matrix(double *cost_matrix, int matrix_size, const char *filename) {
    FILE *file = fopen(filename, "w");
    if (!file) {
        printf("Error: Cannot open %s for writing\n", filename);
        return;
    }
    
    fprintf(file, "%d\n", matrix_size);  // First line: matrix size
    
    // Write matrix (sparse format: only non-zero entries)
    for (int i = 0; i < matrix_size; i++) {
        for (int j = 0; j < matrix_size; j++) {
            double value = cost_matrix[i * matrix_size + j];
            if (value != 0.0) {        // removing the zeros creating sparse matrix 
                fprintf(file, "%d,%d,%.2f\n", i, j, value);
            }
        }
    }
    
    fclose(file);
    printf("Cost matrix exported to %s\n", filename);
}




// Exporting full dense matrix (for debugging)
void export_full_matrix(double *cost_matrix, int matrix_size, const char *filename) {
    FILE *file = fopen(filename, "w");
    if (!file) {
        printf("Error: Cannot open %s for writing\n", filename);
        return;
    }
    
    fprintf(file, "Full Cost Matrix (%d x %d)\n", matrix_size, matrix_size);
    fprintf(file, "================================\n\n");
    
    // Column headers
    fprintf(file, "      ");
    for (int j = 0; j < matrix_size; j++) {
        fprintf(file, "Col%-5d ", j);
    }
    fprintf(file, "\n");
    
    fprintf(file, "      ");
    for (int j = 0; j < matrix_size; j++) {
        fprintf(file, "-------- ");
    }
    fprintf(file, "\n");
    
    // Matrix rows
    for (int i = 0; i < matrix_size; i++) {
        fprintf(file, "Row%-2d | ", i);
        
        for (int j = 0; j < matrix_size; j++) {
            double value = cost_matrix[i * matrix_size + j];
            
            if (value == 0.0) {
                fprintf(file, "    0    ");  // Zero entry
            } else if (i == j) {
            // Diagonal: always show the number (it's fuel cost)
            fprintf(file, "%8.0f ", value);
            } else if (value >= 10000.0) {
            // Off-diagonal: large value = conflict
                fprintf(file, " CONFLICT");
            } else {
                fprintf(file, "%8.0f ", value);  // Regular cost
            }
        }
        fprintf(file, "\n");
    }
    
    fprintf(file, "\n================================\n");
    fprintf(file, "Legend:\n");
    fprintf(file, "  Diagonal entries = Fuel costs (kg)\n");
    fprintf(file, " (0,0) is flight 0 taking route 0 its fuel cost, where rows 0 to 4 is flight routes fro the zeroth flight(AC101), similary col 0 to 4 is allroutes available for Flight 0  \n");
    fprintf(file, "  CONFLICT = Route conflict penalty; No conflicts for now (10000+ kg)\n");
    fprintf(file, "  0 = No interaction\n");
    
    fclose(file);
    printf("Full matrix exported to %s\n", filename);
}

Overwriting astar.c


In [17]:
%%writefile main.c
/**
 * IrisPathQ Route Optimizer
 * Main program - data loading, route generation, and cost matrix creation
*/

#include "data_structures.h"
//function declaration

int main() {
    printf("====================================\n");
    printf("IrisPathQ Route Optimization\n");
    printf("====================================\n\n");

    ProblemInstance problem;
    memset(&problem, 0, sizeof(ProblemInstance));

    printf("Loading data...\n");
    load_flights("data/flights.csv", &problem);
    load_waypoints("data/waypoints.csv", &problem);
    load_weather("data/weather.csv", &problem);

    printf("\n====================================\n");
    printf("Generating Routes\n");
    printf("====================================\n\n");
    
    int num_alternative_routes = 5;
    for (int i = 0; i < problem.num_flights; i++) {
        generate_alternative_routes(&problem, i, num_alternative_routes);
    }
    //Conflict
    inject_conflicts(&problem);


    printf("\n====================================\n");
    printf("Building Cost Matrix\n");
    printf("====================================\n\n");
    
    double *cost_matrix = NULL;
    int matrix_size = 0;
    build_cost_matrix(&problem, &cost_matrix, &matrix_size);
    
    export_cost_matrix(cost_matrix, matrix_size, "output/cost_matrix.txt");
    export_full_matrix(cost_matrix, matrix_size, "output/full_matrix.txt");
    
    printf("\n====================================\n");
    printf("Summary\n");
    printf("====================================\n");
    printf("Total flights: %d\n", problem.num_flights);
    printf("Total routes generated: %d\n", matrix_size);
    printf("Cost matrix size: %d x %d\n", matrix_size, matrix_size);
    printf("\nReady for quantum optimization!\n");
    
    free(cost_matrix);
    return 0;
}



Overwriting main.c


In [10]:
import subprocess

print("Compiling IrisPathQ Route Optimizer...\n")

result = subprocess.run([
    "gcc", "-o", "route_optimizer","main.c", "data_loader.c", "utils.c", "astar.c","-lm"], capture_output=True, text=True)

if result.returncode == 0:
    print("Compilation working\nExecutable: route_optimizer")

else:
    print("Compilation failed:")
    print(result.stderr)

Compiling IrisPathQ Route Optimizer...

Compilation working
Executable: route_optimizer


In [18]:
import subprocess

print("Running IrisPathQ Route Optimizer...\n")
print("=" * 50) #interesting

result = subprocess.run(["./route_optimizer"], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print("Error:", result.stderr)

Running IrisPathQ Route Optimizer...

IrisPathQ Route Optimization

Loading data...
Loaded 5 flights
Loaded 10 waypoints
Loaded 3 weather cells

Generating Routes

Generating 5 routes for AC101: YYZ -> YVR
  Route 0: Route: YYZ -> YVR 
Distance: 1806.46 nm, Fuel: 10035.91 kg, Time: 4.01 hrs
  Route 1: (variant) Route: YYZ -> YVR 
Distance: 1806.46 nm, Fuel: 10236.63 kg, Time: 4.09 hrs
  Route 2: (variant) Route: YYZ -> YVR 
Distance: 1806.46 nm, Fuel: 10437.35 kg, Time: 4.17 hrs
  Route 3: (variant) Route: YYZ -> YVR 
Distance: 1806.46 nm, Fuel: 10638.07 kg, Time: 4.26 hrs
  Route 4: (variant) Route: YYZ -> YVR 
Distance: 1806.46 nm, Fuel: 10838.79 kg, Time: 4.34 hrs
Generating 5 routes for WS202: YUL -> YYC
  Route 0: Route: YUL -> YYC 
Distance: 1622.86 nm, Fuel: 8655.23 kg, Time: 3.61 hrs
  Route 1: (variant) Route: YUL -> YYC 
Distance: 1622.86 nm, Fuel: 8828.33 kg, Time: 3.68 hrs
  Route 2: (variant) Route: YUL -> YYC 
Distance: 1622.86 nm, Fuel: 9001.44 kg, Time: 3.75 hrs
  Route

In [19]:
print("Cost Matrix:\n")
with open("output/cost_matrix.txt", "r") as f:
    lines = f.readlines()
    for line in lines:
        print(line.strip())

Cost Matrix:

25
0,0,10035.91
0,5,10000.00
0,10,10000.00
0,15,10000.00
0,20,10000.00
1,1,10236.63
1,5,10000.00
1,10,10000.00
1,15,10000.00
1,20,10000.00
2,2,10437.35
2,5,10000.00
2,10,10000.00
2,15,10000.00
2,20,10000.00
3,3,10638.07
3,5,10000.00
3,10,10000.00
3,15,10000.00
3,20,10000.00
4,4,10838.79
4,5,10000.00
4,10,10000.00
4,15,10000.00
4,20,10000.00
5,0,10000.00
5,1,10000.00
5,2,10000.00
5,3,10000.00
5,4,10000.00
5,5,8655.23
5,10,10000.00
5,15,10000.00
5,20,10000.00
6,6,8828.33
7,7,9001.44
8,8,9174.54
9,9,9347.65
10,0,10000.00
10,1,10000.00
10,2,10000.00
10,3,10000.00
10,4,10000.00
10,5,10000.00
10,10,8547.33
10,15,10000.00
10,20,10000.00
11,11,8718.28
12,12,8889.22
13,13,9060.17
14,14,9231.12
15,0,10000.00
15,1,10000.00
15,2,10000.00
15,3,10000.00
15,4,10000.00
15,5,10000.00
15,10,10000.00
15,15,7729.46
15,20,10000.00
16,16,7884.04
17,17,8038.63
18,18,8193.22
19,19,8347.81
20,0,10000.00
20,1,10000.00
20,2,10000.00
20,3,10000.00
20,4,10000.00
20,5,10000.00
20,10,10000.00
20,15,100

In [20]:
print("Full Matrix Visualization (first 30 lines only ):\n")
with open("output/full_matrix.txt", "r") as f:
    lines = f.readlines()[:30]
    for line in lines:
        print(line.rstrip())

Full Matrix Visualization (first 30 lines only ):

Full Cost Matrix (25 x 25)

      Col0     Col1     Col2     Col3     Col4     Col5     Col6     Col7     Col8     Col9     Col10    Col11    Col12    Col13    Col14    Col15    Col16    Col17    Col18    Col19    Col20    Col21    Col22    Col23    Col24
      -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- -------- --------
Row0  |    10036     0        0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0
Row1  |     0       10237     0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0     CONFLICT    0        0        0        0
Ro

In [24]:

%%writefile route_qaoa.py

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit import Parameter
import sys
import time

print("=" * 70)
print("OPTIMIZED 5-FLIGHT QAOA")
print("=" * 70)

# FIRST: Load cost matrix from C cost_matrix.txt
print("\nLoading cost matrix...")
try:
    with open('output/cost_matrix.txt', 'r') as f:
        lines = f.readlines()
    
    # As First line contains matrix size
    matrix_size = int(lines[0].strip())

    #Intialize Empty matrix 
    cost_matrix = np.zeros((matrix_size, matrix_size))

    # Tracking what  loading
    num_diagonal = 0
    num_conflicts = 0
    total_conflict_penalty = 0
    
    #Now parse the SPARSE matrix ; format: "row,col,value"
    for line in lines[1:]:
        parts = line.strip().split(',')
        if len(parts) == 3:
            i, j, value = int(parts[0]), int(parts[1]), float(parts[2])
            cost_matrix[i][j] = value

            # Track what we loaded
            if i == j:
                num_diagonal += 1
            else:
                if value >= 10000:  # Conflict penalty
                    num_conflicts += 1
                    total_conflict_penalty += value
    
    print(f"Loaded {matrix_size}*{matrix_size} matrix")
    print(f"  Diagonal entries (fuel costs): {num_diagonal}")
    print(f"  Off-diagonal entries (conflicts): {num_conflicts}")
    print(f"  Total conflict penalty in matrix: {total_conflict_penalty:.0f} kg")

    if num_conflicts == 0:
        print("WARNING: No conflicts found in matrix!")
    else:
        print(f"Conflicts detected - quantum advantage possible!")
    
    # Always run C first(to create the matrix) ; weather will be taken into account before creating the matrix 
except FileNotFoundError:
    print("ERROR: Run ./simulator.o first!")
    sys.exit(1)


# SECOND: Determine structure of the matrix 
# With 5 flights and 5 routes per flight, we have 25 total variables.
# Variables are grouped: indices 0-4 = Flight 0 routes, 5-9 = Flight 1, etc

NUM_FLIGHTS = 5
ROUTES_PER_FLIGHT = matrix_size // NUM_FLIGHTS

print(f"Flights: {NUM_FLIGHTS}, Routes/flight: {ROUTES_PER_FLIGHT}")


# Third: Show fuel costs
print("\nFuel costs (kg):")

for i in range(matrix_size):
    flight = i // ROUTES_PER_FLIGHT
    route = i % ROUTES_PER_FLIGHT
    print(f"  Flight {flight} Route {route}: {cost_matrix[i][i]:.0f}")

#Fourth:  Classical greedy
# For comparison, for now using a greedy approach:
# Each flight independently selects its cheapest route (no global optimization)
print("\n" + "=" * 70)
print("CLASSICAL SOLUTION")
print("-" * 70)

classical_solution = []
for flight in range(NUM_FLIGHTS):
     # Get route indices for this flight
    start = flight * ROUTES_PER_FLIGHT
    end = start + ROUTES_PER_FLIGHT
    
    # Find cheapest route for this flight
    best_route = min(range(start, end), key=lambda r: cost_matrix[r][r])
    classical_solution.append(best_route)

# Calculate total fuel cost
classical_fuel = sum(cost_matrix[r][r] for r in classical_solution)

# Calculate conflict penalties (off-diagonal)
classical_conflicts = 0
for i in range(len(classical_solution)):
    for j in range(i + 1, len(classical_solution)):
        classical_conflicts += cost_matrix[classical_solution[i]][classical_solution[j]]

classical_cost = classical_fuel + classical_conflicts #sum(cost_matrix[r][r] for r in classical_solution)

#Display
for i, route_idx in enumerate(classical_solution):
    print(f"  Flight {i}: Route {route_idx % ROUTES_PER_FLIGHT} ({cost_matrix[route_idx][route_idx]:.0f} kg)")
    print(f"Total: {classical_cost:.0f} kg")

print(f"\n  Fuel costs:         {classical_fuel:.0f} kg")
print(f"  Conflict penalties: {classical_conflicts:.0f} kg")
print(f"  TOTAL COST:         {classical_cost:.0f} kg")

if classical_conflicts > 0:
    num_conflicts = int(classical_conflicts / 10000)  # for now 10000 kg per conflict
    print(f"\n Greedy picked {num_conflicts} conflicting route pairs!")

# Fifth: Quantum - simplified 
print("\n" + "=" * 70)
print("QAOA SOLUTION")
print("-" * 70)



# Design Circuit 
# We use 2 qubits per flight to encode route choice:
# 2 qubits can represent 4 states: 00, 01, 10, 11 (routes 0-3)
# With 5 flights × 2 qubits = 10 total qubits
# Use 2 qubits per flight (can represent 0,1,2,3 = 4 routes)
QUBITS_PER_FLIGHT = 2
num_qubits = NUM_FLIGHTS * QUBITS_PER_FLIGHT

print(f"TotalQubits: {num_qubits}")

qc = QuantumCircuit(num_qubits, num_qubits)

# Simplified circuit for speed; to scan through all settings
#e.g how much to listen and how much to explore two params 
#Larger γ = Circuit pays MORE attention to minimizing fuel (rz gate)
#Larger β = More exploration (quantum tries more combinations)(rx gate)
gamma = Parameter('γ')
beta = Parameter('β')

#Now QAOA layer 
# Superposition: So all possible route combinations
for q in range(num_qubits):
    qc.h(q)

# Cost encoding (simplified)[Hamiltonian]
# The constructive interference using RZ gate applies phase rotation to fuel cost
for flight in range(NUM_FLIGHTS):
    q = flight * QUBITS_PER_FLIGHT
    qc.rz(gamma, q)

# Mixing [tunneling]
# using RX gate  fro tunneling between diff route choices 
for q in range(num_qubits):
    qc.rx(beta, q)

#measurement collapse all into bitstring; classical 
qc.measure_all()

print(f"Circuit depth: {qc.depth()}, gates: {qc.size()}")

# Finally, Run simulation
print("Running quantum simulation...")
start_time = time.time()

#Qiskit's built iin simulator
simulator = Aer.get_backend('qasm_simulator')

best_solution = None
best_cost = float('inf')

# Single parameter set for speed, MIGHT not be optimal just simple values to test
# in future use **variational optimization approach**
bound_qc = qc.assign_parameters({gamma: 1.0, beta: 0.5})

# Compile circuit for simulator
compiled = transpile(bound_qc, simulator, optimization_level=1)

job = simulator.run(compiled, shots=500)  #  shots 
result = job.result()
counts = result.get_counts()  #bitstring → count

elapsed = time.time() - start_time
print(f"Completed in {elapsed:.1f} seconds")

# Decode top 10 results; Sorted by frequency (most common measurements first)
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

#only top 10
for bitstring, count in sorted_counts[:10]:
    solution = []
    

    # Parse bitstring to extract route choice for each flight
    for flight in range(NUM_FLIGHTS):
        bit_offset = flight * QUBITS_PER_FLIGHT      # Extract bits for this flight

        #bit odering 
        flight_bits = bitstring[-(bit_offset + QUBITS_PER_FLIGHT):][:QUBITS_PER_FLIGHT]
        
        # Convert binary to route index (0-3)
        route_choice = int(flight_bits[::-1], 2) % ROUTES_PER_FLIGHT

        # Calculate global route index
        route_idx = flight * ROUTES_PER_FLIGHT + route_choice
        solution.append(route_idx)

    # Calculate total fuel cost for this solution
    cost = sum(cost_matrix[r][r] for r in solution)
    
    # Track best solution found
    if cost < best_cost:
        best_cost = cost
        best_solution = solution

# FINALLY, Display Best Quantum Solution
print("\nQuantum solution:")
if best_solution:
    for i, route_idx in enumerate(best_solution):
        print(f"  Flight {i}: Route {route_idx % ROUTES_PER_FLIGHT} ({cost_matrix[route_idx][route_idx]:.0f} kg)")
    print(f"Total: {best_cost:.0f} kg")
else:
    print("  No solution found")

# Comparison
print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)
print(f"Classical: {classical_cost:.0f} kg")
print(f"Quantum:   {best_cost:.0f} kg")

if best_cost <= classical_cost:
    print("SUCCESS: Quantum matched or beat classical!")
else:
    diff = best_cost - classical_cost
    print(f"Classical better by {diff:.0f} kg (quantum needs tuning)")

print("\n" + "=" * 70)

Overwriting route_qaoa.py
